In [25]:
from bs4 import BeautifulSoup
from bs4 import Tag
import json
import os
import tqdm
from selenium.webdriver.chrome.service import Service
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
import requests

In [26]:
l=[
    ("https://www.aparat.com/v/jsPIg?playlist=1041759","dars_hafez.html",'درس‌گفتار حافظ‌شناسی'),
    ("https://www.aparat.com/v/tRhOI?playlist=6411553","fekr_saeb.html",'فکر صائب'),
    ("https://www.aparat.com/v/yphvH?playlist=881126","jam_hafez.html",'جامِ جهان‌نمایِ حافظ'),
]

In [27]:
def parse_item(item:Tag):
    prefix="https://www.aparat.com"
    poster=item.select_one('.poster img')['src']
    vid_url=item.select_one('.thumb-content a')['href']
    title=item.select_one('.title span').text
    return poster,prefix+vid_url,title

In [28]:
links=[]

In [71]:
def get_download_link(link:str):
    print(link)
    video_id=link.split('\\')[-1].split('/')[-1].split("?")[0]
    print(video_id)
    api_url = f"https://www.aparat.com/etc/api/video/videohash/{video_id}/get"

    chrome_driver_path = "C:\\chromedriver.exe"
    s=Service(executable_path=chrome_driver_path)
    # Launch Chrome with the driver
    driver = webdriver.Chrome(service=s)

    # Navigate to a website
    driver.get(api_url)
    print(driver.page_source)
    content=driver.find_element(By.TAG_NAME,'pre').text
    # response = requests.get(api_url)
    # data = response.json()
    data = json.loads(content)

    with open("tmp.json",'w',encoding='utf-8')as f:
        f.write(driver.page_source)
    if data['video']:
        videos = data["video"]["file_link_all"]
        for i in videos:
            if i['profile']=='720p':return i['urls'][0]
        else:
            return videos[-1]['urls'][0]
    return ""

In [72]:
def html_to_parsed_json(file_name,output_file,category):
    l=[]
    os.makedirs("jsons",exist_ok=True)
    with open(file_name,encoding="utf8") as fp:
        soup = BeautifulSoup(fp, "html.parser")
    items=soup.select('.item-playlist')
    for i in items:
        poster,vid_url,title=parse_item(i)
        vid_down_url=get_download_link(vid_url)
        d={"poster":poster,"vid_url":vid_url,"title":title,"info":"","vid_down_url":vid_down_url}
        f_n=file_name.split('/')[-1].split("\\")[-1].split('.')[0].replace('_','-')
        l.append({"data":d,"category":category,"category_fing":f_n})
    of=output_file if output_file.endswith('.json') else output_file+".json"
    with open(os.path.join('jsons',of), "w",encoding='utf8') as f:
        json.dump(l,f,ensure_ascii=False)

In [73]:
for i in l:
    i=i[1:]
    f_n=os.path.join('htmls',i[0])
    j_name=i[0].replace('.html','.json')
    category=i[1]
    html_to_parsed_json(f_n,j_name,category)

https://www.aparat.com/v/jsPIg?playlist=1041759
jsPIg
<html><head><meta name="color-scheme" content="light dark"></head><body><pre style="word-wrap: break-word; white-space: pre-wrap;">{"video":{"id":"34776245","title":"1- \u062f\u0631\u0633\u200c\u06af\u0641\u062a\u0627\u0631\u0650 \u062d\u0627\u0641\u0638\u200c\u0634\u0646\u0627\u0633\u06cc\u061b \u0628\u0627\u0632\u0622\u06cc \u0648 \u062f\u0644\u0650 \u062a\u0646\u06af\u0650 \u0645\u0631\u0627 \u0645\u0648\u0646\u0633\u0650 \u062c\u0627\u0646 \u0628\u0627\u0634","username":"Farhangi_iust","userid":"7497212","visit_cnt":1056,"uid":"jsPIg","isHidden":false,"process":"done","sender_name":"\u0627\u0646\u062c\u0645\u0646 \u0639\u0644\u0645\u06cc \u0634\u0627\u0631\u062d","big_poster":"https:\/\/static.cdn.asset.aparat.com\/avt\/34776245-4886-b__6232.jpg","small_poster":"https:\/\/static.cdn.asset.aparat.com\/avt\/34776245-4886__6232.jpg","profilePhoto":"https:\/\/static.cdn.asset.aparat.com\/profile-photo\/7497212-4886-m.jpg","duration"

In [74]:
conc_l=[]
for i in os.listdir("jsons/"):
    with open(os.path.join("jsons",i) ,encoding='utf-8') as f:
        obj=json.load(f)
        conc_l+=(obj)
with open("result.json", "w",encoding='utf8') as f:
    json.dump(conc_l,f,ensure_ascii=False)        

In [75]:
conc_l.__len__()

6